# Failure analyst

One conversation over every flagged memory in NAMS, oldest first, numbered from 0. **Load** uses the same checkpoint list as play. The analyst loads Gemma only; S1 stays unused. Neo4j has to be up.

The reply can end with one tool token. The notebook shows the result and the model continues, up to 24 tools, then the turn comes back.

- `[LIST]` — one line per flag: number, game, flag text
- `[LOAD n]` — both questions, both messages, the before picture and its Settings
- `[LOAD n: parts]` — only those parts. `bq bm bi bs aq am ai as gaze`. `b`/`a` is before/after; `q` question, `m` message, `i` image, `s` settings
- `[DROP n]`, `[DROP n: parts]`, `[DROP ALL]`
- `[GAZE n]` — s and v on the before message: at the start, at the end of each line, and at the final `]`
- `[GAZE n: a-b]` — every token from a to b. `[GAZE n: -k]` is the last k tokens
- `[NOTE key: text]` and `[FORGET key]` — the scratchpad, not a tip
- `[SEARCH words]` — flagged messages whose text matches

At most 4 pictures stay loaded. A loaded memory stays until it is dropped. **Restart thread** starts a new analyst conversation and clears the loaded set. Floats on screen are three decimals. Stored traces are not rewritten.


In [ ]:
import os
import sys
import traceback
from pathlib import Path

if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())

import ipywidgets as widgets

from agent.analyst import FailureAnalyst
from neural_net.paths import weights_root
from neural_net.weight_catalog import (
    label as catalog_label,
    list_assembled,
    list_coord,
    list_gemma,
    list_localizer,
    list_s1,
)

state = {"analyst": None, "busy": False}


def _options(paths, empty):
    if not paths:
        return [(empty, "")]
    return [(catalog_label(Path(path)), str(path)) for path in paths]


def _keep(dropdown, options):
    selected = dropdown.value
    dropdown.options = options
    values = [value for _, value in options]
    dropdown.value = selected if selected in values else values[0]


def _gemma_dir(mode):
    if mode == "full":
        root = Path(full_dd.value)
        child = root / "gemma"
        return str(child if child.is_dir() else root)
    if mode == "parts":
        return gemma_dd.value
    return coord_dd.value


def _say(text):
    log.append_stdout(text if text.endswith("\n") else text + "\n")


def _set_busy(flag):
    state["busy"] = flag
    for widget in controls:
        widget.disabled = flag


def _refresh_side():
    analyst = state["analyst"]
    if analyst is None:
        loaded.value = "<pre>Nothing loaded.</pre>"
        pad.value = "<pre>No thread.</pre>"
        thumbs.children = ()
        return
    loaded.value = "<pre>" + analyst.loaded_line() + "</pre>"
    pad.value = "<pre>" + analyst.notepad() + "</pre>"
    children = []
    for item in analyst.loaded_images():
        file = Path(item["path"])
        if not file.is_file():
            raise FileNotFoundError(f"loaded image is missing: {file}")
        children.append(widgets.VBox([
            widgets.HTML(
                f"<b>memory {item['n']} {item['side']}</b>"
            ),
            widgets.Image(
                value=file.read_bytes(), format="png", width=240,
            ),
        ]))
    thumbs.children = tuple(children)


def on_refresh(_):
    if state["busy"]:
        return
    _keep(full_dd, _options(list_assembled(), "(no full save)"))
    _keep(gemma_dd, _options(list_gemma(), "(no Gemma)"))
    _keep(loc_dd, _options(list_localizer(), "(no localizer)"))
    _keep(coord_dd, _options(list_coord(), "(no coord checkpoint)"))
    _keep(s1_dd, _options(list_s1(), "(no S1 checkpoint)"))
    _say(f"weights under {weights_root()}")


def on_load(_):
    if state["busy"]:
        return
    path = _gemma_dir(mode_dd.value)
    if not path:
        status.value = "Pick a checkpoint."
        return
    _set_busy(True)
    status.value = "Loading."
    try:
        if state["analyst"] is None:
            state["analyst"] = FailureAnalyst(load_model=False)
        info = state["analyst"].load_gemma_dir(path)
        status.value = "Loaded " + str(info["key"]) + "."
        _say(f"checkpoint {path}")
        _refresh_side()
    except Exception:
        status.value = "Load failed."
        _say(traceback.format_exc())
    finally:
        _set_busy(False)


def on_ask(_):
    if state["busy"]:
        return
    analyst = state["analyst"]
    if analyst is None or analyst.model is None:
        status.value = "Load a checkpoint first."
        return
    question = box.value.strip()
    if not question:
        status.value = "Write a question."
        return
    _set_busy(True)
    status.value = "Asking."
    chunks = []

    def _step(step):
        if step.get("text"):
            chunks.append(step["text"])
            reply.value = "<pre>" + "\n\n".join(chunks) + "</pre>"
        if step.get("tool_result") is not None:
            chunks.append(step["tool_result"])
            reply.value = "<pre>" + "\n\n".join(chunks) + "</pre>"
            _refresh_side()

    try:
        analyst.ask(question, on_step=_step)
        status.value = "Done."
        _refresh_side()
    except Exception:
        status.value = "Ask failed."
        _say(traceback.format_exc())
    finally:
        _set_busy(False)


def on_restart(_):
    if state["busy"]:
        return
    analyst = state["analyst"]
    if analyst is None:
        status.value = "Load a checkpoint first."
        return
    info = analyst.restart()
    reply.value = "<pre></pre>"
    status.value = "New thread."
    _say("thread " + info["session_id"])
    _refresh_side()


mode_dd = widgets.Dropdown(
    options=[
        ("Full save", "full"),
        ("Gemma from here, localizer from here", "parts"),
        ("Coord checkpoint", "coord"),
    ],
    description="Load:",
    style={"description_width": "70px"},
)
full_dd = widgets.Dropdown(
    options=_options(list_assembled(), "(no full save)"), description="Full",
)
gemma_dd = widgets.Dropdown(
    options=_options(list_gemma(), "(no Gemma)"), description="Gemma",
)
loc_dd = widgets.Dropdown(
    options=_options(list_localizer(), "(no localizer)"), description="Localizer",
)
coord_dd = widgets.Dropdown(
    options=_options(list_coord(), "(no coord checkpoint)"), description="Coord",
)
s1_dd = widgets.Dropdown(
    options=_options(list_s1(), "(no S1 checkpoint)"), description="S1",
)
refresh_btn = widgets.Button(description="Refresh")
load_btn = widgets.Button(description="Load", button_style="primary")
box = widgets.Textarea(
    value="What do these flags have in common?",
    layout=widgets.Layout(width="100%", height="80px"),
)
ask_btn = widgets.Button(description="Ask", button_style="primary")
restart_btn = widgets.Button(description="Restart thread")
status = widgets.HTML("<span>Load a checkpoint.</span>")
reply = widgets.HTML("<pre></pre>")
loaded = widgets.HTML("<pre>Nothing loaded.</pre>")
pad = widgets.HTML("<pre>No thread.</pre>")
thumbs = widgets.HBox()
log = widgets.Output(layout=widgets.Layout(height="120px", overflow="auto"))

controls = [
    mode_dd, full_dd, gemma_dd, loc_dd, coord_dd, s1_dd,
    refresh_btn, load_btn, box, ask_btn, restart_btn,
]
refresh_btn.on_click(on_refresh)
load_btn.on_click(on_load)
ask_btn.on_click(on_ask)
restart_btn.on_click(on_restart)

widgets.VBox([
    mode_dd, full_dd, gemma_dd, loc_dd, coord_dd, s1_dd,
    widgets.HBox([refresh_btn, load_btn]),
    status,
    box,
    widgets.HBox([ask_btn, restart_btn]),
    widgets.HTML("<b>Reply</b>"),
    reply,
    widgets.HTML("<b>Loaded</b>"),
    loaded,
    widgets.HTML("<b>Scratchpad</b>"),
    pad,
    widgets.HTML("<b>Loaded images</b>"),
    thumbs,
    log,
])
